STEP 0: Bring Groq's llm

In [11]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

STEP 1: Extract Text from new PDF

In [12]:
# extracts text from pdf and make into langchain Documents (one page -> one Document)
# pypdf itself creates the metadata for each Document

from langchain_community.document_loaders import PyPDFLoader 

loader = PyPDFLoader("../docs/Rithish_S_LG_Ad_Solutions_Cover_Letter.pdf") # new pdf's content 

docs = loader.load()

STEP 2: Convert the documents into chunks

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_documents(docs)

STEP 3: Create embedding model

In [14]:
from langchain_huggingface import HuggingFaceEmbeddings 

embedding_model = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2") 

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5835.26it/s]


STEP 4: Reuse the vector database, by skipping step 2 and 3

In [15]:
from langchain_chroma import Chroma

vector_store_persistant = Chroma(
    persist_directory='../vector_db',
    embedding_function=embedding_model # to make sure the prompt sent to chromadb also to use the same embedding model
)

STEP 5: Add new Pdf to existing local chromaDB

In [16]:
vector_store_persistant.add_documents(chunks)

['e63af038-c046-456a-8639-b062592c4542',
 'ae13f93e-bf5d-4dc1-8540-3a60b439c66e',
 'e54d9bb1-e8d9-45ac-b1b5-ecc1027b871b',
 '618d91d1-283d-4fc8-ba75-15e2a51d7c32',
 '1c4212b6-f1eb-493a-978f-d66ed9829d40',
 '8f484f2c-f41d-4abe-9197-a1fd89a9cce2',
 '4e6f4f31-43a7-4f21-858b-89bfdac17ae4']

STEP 6: Retrive top 3 chunks

In [17]:
context = vector_store_persistant.similarity_search("Who is rithish and his interests ?") 

STEP 7: Provide context to llm to answer the user query

In [18]:
llm.invoke(f"Who is rithish and his interests ? You can answer using following context: {context}")

AIMessage(content='**Rithish S** is a Computer Science and Engineering student at Bannari\u202fAmman\u202fInstitute of Technology, set to graduate in 2028.  \n\n**Interests:**  \n- Backend engineering  \n- Distributed systems  \n- Scalable software development  \n\nThese interests are highlighted in his cover letter for the Software Engineer\u202fI position at LG Ad Solutions.', additional_kwargs={'reasoning_content': 'We need to answer: "Who is rithish and his interests?" Use context. The documents show a cover letter by Rithish S, a Computer Science and Engineering student at Bannari Amman Institute of Technology, graduating 2028, with strong interest in backend engineering, distributed systems, and scalable software development. So answer accordingly.'}, response_metadata={'token_usage': {'completion_tokens': 151, 'prompt_tokens': 1044, 'total_tokens': 1195, 'completion_time': 0.161123962, 'completion_tokens_details': {'reasoning_tokens': 70}, 'prompt_time': 0.077107432, 'prompt_tok